# Time varying functions

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `detailed/time-varying-functions` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

While it is possible to construct arbitrary time-varying rates for use in
summer4 models, a few cases are common enough for convenience helpers:
linear and sigmoidal interpolation of sparse knots, piecewise-constant
steps, and composition of those into a flow rate.


In [ ]:
from typing import NamedTuple

import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    EntryFlow,
    FlowModel,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Time,
    TransitionFlow,
    derived_refs,
)
from summer4.timevarying import linear, piecewise, sigmoidal, step

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("Y",))
pmap = PropertyMap.from_property(state)
y0 = PropertyData.wrap(pmap, np.array([0.0]))


def eval_rate(rate_expr, times, params=None):
    model = FlowModel(pmap)
    model.add_flow(EntryFlow("in", state["Y"], rate_expr))
    cm = model.compile()
    p = {} if params is None else params
    return np.array([float(np.asarray(cm.vector_field(t, y0, p).data)[0]) for t in times])


## Linear interpolation

`linear(arg, breakpoints, values)` builds a structural `Interp` node.
Outside the knot range it clamps to the end values — it does not extrapolate.


In [ ]:
x_points = np.array((0.0, 1.0, 2.0))
y_points = x_points**2.0
expr = linear(Time(), tuple(x_points), tuple(y_points))

tvals = np.linspace(-1.0, 3.0, 101)
yvals = eval_rate(expr, tvals)
np.testing.assert_allclose(eval_rate(expr, x_points), y_points)
np.testing.assert_allclose(yvals[0], y_points[0])
np.testing.assert_allclose(yvals[-1], y_points[-1])
pd.Series(yvals, index=tvals).plot(labels={"index": "t", "value": "rate"})


## Calibratable knot values (and times)

Knot heights and breakpoint times may both be `FieldRef`s from
`derived_refs`. The breakpoint *count* is fixed; evaluated positions must
stay strictly increasing.


In [ ]:
class Knots(NamedTuple):
    inflection_value: float


refs = derived_refs(Knots)
f_param = linear(Time(), (0.0, 5.0, 10.0), (0.0, refs.inflection_value, 0.0))
grid = np.linspace(0.0, 10.0, 11)
out_hi = eval_rate(f_param, grid, Knots(inflection_value=2.2))
out_lo = eval_rate(f_param, grid, Knots(inflection_value=-0.1))
np.testing.assert_allclose(out_hi[5], 2.2, atol=1e-5)
np.testing.assert_allclose(out_lo[5], -0.1, atol=1e-5)
print("parameterized midpoint", float(out_hi[5]), float(out_lo[5]))


class KnotXY(NamedTuple):
    inflection_time: float
    inflection_value: float


xy = derived_refs(KnotXY)
f_xy = linear(
    Time(),
    (0.0, xy.inflection_time, 10.0),
    (0.0, xy.inflection_value, 0.0),
)
in_domain = np.linspace(0.0, 10.0, 101)
frame = pd.DataFrame(index=in_domain)
for t_mid, h in ((1.0, 2.2), (9.0, -0.5)):
    frame[f"t={t_mid},h={h}"] = eval_rate(
        f_xy, in_domain, KnotXY(inflection_time=t_mid, inflection_value=h)
    )
np.testing.assert_allclose(
    eval_rate(f_xy, [1.0], KnotXY(inflection_time=1.0, inflection_value=2.2))[0],
    2.2,
    atol=1e-5,
)
np.testing.assert_allclose(
    eval_rate(f_xy, [9.0], KnotXY(inflection_time=9.0, inflection_value=-0.5))[0],
    -0.5,
    atol=1e-5,
)
frame.plot(labels={"index": "t", "value": "rate"})


## Sigmoidal interpolators

`sigmoidal(..., sharpness=...)` uses summer2's curvature semantics: `1.0` is
linear-equivalent after normalization; larger values approach a step at each
segment midpoint. Each segment stays within its knot bounds.


In [ ]:
x_sig = (0.0, 1.0, 2.0, 3.0, 4.0)
y_sig = (0.0, 1.0, -2.0, 0.5, 3.0)
in_domain = np.linspace(0.0, 4.0, 101)
frame = pd.DataFrame(index=in_domain)
for sharpness in (1.0, 8.0, 16.0, 128.0):
    frame[str(sharpness)] = eval_rate(
        sigmoidal(Time(), x_sig, y_sig, sharpness=sharpness), in_domain
    )
frame.plot(labels={"index": "t", "value": "rate"})
np.testing.assert_allclose(
    eval_rate(sigmoidal(Time(), x_sig, y_sig, sharpness=16.0), list(x_sig)),
    y_sig,
    atol=1e-4,
)


## Piecewise / step functions

`step` (aliased as `piecewise`) takes `len(values) == len(breakpoints) + 1`.
At a breakpoint the new (right) value is taken — right-continuous.


In [ ]:
f_step = step(Time(), (0.0, 1.0), (-1.0, 0.0, 1.0))
assert piecewise(Time(), (0.0,), (1.0, 2.0)).kind == "step"
domain = np.linspace(-1.0, 2.0, 101)
out = eval_rate(f_step, domain)
np.testing.assert_allclose(eval_rate(f_step, [0.0]), [0.0])
np.testing.assert_allclose(eval_rate(f_step, [-0.5]), [-1.0])
pd.Series(out, index=domain).plot(labels={"index": "t", "value": "rate"})


## Composition and use in a model

Overlay a zero window on a linear ramp with `piecewise`, then drive an SIR
infection flow with that schedule. (A force-of-infection primitive is WP6;
here the schedule is the fractional infection rate itself.)


In [ ]:
baseline = linear(Time(), (0.0, 10.0), (0.0, 1.0))
overlay = piecewise(Time(), (4.0, 5.0), (baseline, 0.0, baseline))
comp_t = np.linspace(0.0, 10.0, 100)
comp_y = eval_rate(overlay, comp_t)
np.testing.assert_allclose(eval_rate(overlay, [4.5]), [0.0], atol=1e-6)
assert eval_rate(overlay, [2.0])[0] > 0.0
pd.Series(comp_y, index=comp_t).plot(labels={"index": "t", "value": "rate"})


In [ ]:
sir = Property("state", ("S", "I", "R"))
spmap = PropertyMap.from_property(sir)


class Rates(NamedTuple):
    contact_rate: float


D = derived_refs(Rates)
model = FlowModel(spmap)
model.add_flow(TransitionFlow("infection", sir["S"], sir["I"], overlay * D.contact_rate))
model.add_flow(TransitionFlow("recovery", sir["I"], sir["R"], 1.0))
cm = model.compile()
sy0 = PropertyData.wrap(spmap, np.array([100.0, 1.0, 0.0]))
plan = SavePlan(requests={"compartments": SaveRequest(Compartments())})
res = cm.run(
    Rates(contact_rate=10.0), sy0, t0=0.0, t1=10.0, dt=0.1, save=plan, solver="tsit5"
)
pdf = res["compartments"].to_pandas()
i = np.asarray(res["compartments"].select(sir["I"]).values.data).reshape(-1)
assert float(i[-1]) > 0.0
pdf.plot(labels={"index": "time", "value": "people"})
